### Binning Metagenomes

- clustering-based binning
- Grouping contigs based on similar sequences into bins...essentially creating metagenome-assembled metagenomes (MAGs). The goal is that these contigs (sequences) will come together to form a single genome that represents a microbial species. These MAGs can be matched to a reference database and a species can be assigned, or the MAG represents a novel species.  

Software documentations:

MetaBAT: https://bitbucket.org/berkeleylab/metabat/src/master/README.md

CONCOCT: https://github.com/BinPro/CONCOCT

CheckM: https://github.com/Ecogenomics/CheckM/wiki

Das_tool: https://github.com/cmks/DAS_Tool

In [ ]:
#Create new env and install software 
# conda create -n binning python=3.7
# conda activate binning
# conda install -c bioconda metabat2

# conda install -c bioconda checkm-genome
# conda install -c bioconda das_tool

#### MetaBat2
The first piece of code here generates a fairly simple text file for the coverage of these files. The next set of code runs MetaBat2  (v2.10.2) using minContig 1500, minCV 1.0, minCVSum 1.0, maxP 95%, minS 60, lcuster size 50000 and maxEdges 200. It sets the minimum size for a bin to 200000 basepairs, which is fairly low, so you can keep it. It gathers all mapping information into a single depth file, so you can use your 1 file in the next analysis. An important parameter to play around with is the minimum bin size. When set to 200000, this will severely limit the amount of bins you gain, especially if your samples aren't perfect. Therefore, it is wise to run MetaBAT several times with slight alterations to the -s flag to find your optimal setting (you don't want 3 bins, you also don't want 1000).

For a reference on how to do this accurately, use: https://bitbucket.org/berkeleylab/metabat/wiki/Best%20Binning%20Practices

In [ ]:
## can probably do all binning programs together??
# array job - individual groups 

In [ ]:
# test with mcav first - combined binning script

In [ ]:
# for below 

tail -n +2 "$SAMPLE_FILE"| cut -f 3 |sort| uniq| nl -b a | grep "MCAV"
# testing for MCAV First: array job #s will be: 
     2  122022_MCAV_Diseased_Margin
     3  122022_MCAV_Diseased_Tissue
     4  122022_MCAV_Healthy
    17  52022_MCAV_Diseased_Margin
    18  52022_MCAV_Diseased_Tissue
    19  52022_MCAV_Healthy
    28  62019_MCAV_Healthy

# testing obtaining SPP
GROUPTEST='122022_MCAV_Diseased_Margin'
SPP=$(grep $GROUPTEST "$SAMPLE_FILE" | awk '{print $2}' | uniq)


# set array to all groups in sbatch script - but when submitting, do as follows:
sbatch --array=2-4,17-19,28 /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/binning.sh

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=64G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 72:00:00  # Job time limit
#SBATCH --array=1-31%10
#SBATCH --mail-type=ALL --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/binning/slurm-binning-%A-%a.out

#set parameters for binning:
    # array jobs - per group
SAMPLE_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_sample_groups.txt"
INPUT_GROUP=$(tail -n +2 "$SAMPLE_FILE"| cut -f 3 |sort| uniq | sed -n "${SLURM_ARRAY_TASK_ID}p")
SPP=$(grep "$INPUT_GROUP" "$SAMPLE_FILE" | awk '{print $2}' | uniq)

CONTIGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}"
CONTIGFILE="${INPUT_GROUP}.contigs-fixed.fsa"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/${INPUT_GROUP}/megahit_host_removed"
F_READS="${INPUT_GROUP}_all_reads_R1.fastq.gz"
R_READS="${INPUT_GROUP}_all_reads_R2.fastq.gz"

BAMPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}"
METABINDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/binning/${INPUT_GROUP}/MetaBAT2_bins"
MAXBINDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/binning/${INPUT_GROUP}/Maxbin2_bins"
CONCBINDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/binning/${INPUT_GROUP}/Concoct_bins"
CTEMPDIR="${CONCBINDIR}/concoct_temp"

mkdir -p $METABINDIR
mkdir -p $MAXBINDIR
mkdir -p $CONCBINDIR
mkdir -p $CTEMPDIR

##### run CONCOCT #####
module load conda/latest
conda activate concoct_env

#creates the CONCOCT depth file 
cut_up_fasta.py "$CONTIGPATH/$CONTIGFILE" \
                -c 10000 \
                -o 0 \
                --merge_last \
                -b "$CONCBINDIR/${INPUT_GROUP}_contigs_cut.bed" > "$CONCBINDIR/${INPUT_GROUP}_contigs_cut.fa"

#estimate contig coverage
concoct_coverage_table.py $CONCBINDIR/${INPUT_GROUP}_contigs_cut.bed ${BAMPATH}/*.bam > $CONCBINDIR/coverage_table_"${INPUT_GROUP}".tsv || { echo 'Exit code 2: failed to create coverage file, exiting.' && exit; }

#run CONCOCT
concoct --composition_file $CONCBINDIR/"${INPUT_GROUP}"_contigs_cut.fa \
        --coverage_file $CONCBINDIR/coverage_table_"${INPUT_GROUP}".tsv \
        -t "$SLURM_CPUS_PER_TASK" \
        -b $CTEMPDIR || { echo 'Exit code 3: CONCOCT failed to run, exiting.' && exit; }
merge_cutup_clustering.py $CTEMPDIR/clustering_gt1000.csv > $CTEMPDIR/"${INPUT_GROUP}"_clustering_merged.csv || { echo 'Exit code 4: failed to merge clusters, exiting.' && exit; }
extract_fasta_bins.py $CONTIGPATH/$CONTIGFILE $CTEMPDIR/"${INPUT_GROUP}"_clustering_merged.csv \
                      --output_path $CONCBINDIR || { echo 'Exit code 5: Bins were not extracted, exiting.' && exit; }

conda deactivate
echo "deactivating concoct env"

##### run MetaBAT2 #####
module load conda/latest
conda activate binning
echo "activating metabat2 env"
#create depth file for MetaBat2
jgi_summarize_bam_contig_depths --outputDepth $METABINDIR/MetaBAT2_depth.txt $BAMPATH/*.bam

#MetaBat2 script with verbose output, minimum length (m)(has to be >=1500) and no min bin size 
metabat2 -i $CONTIGPATH/$CONTIGFILE \
         -a $METABINDIR/MetaBAT2_depth.txt \
         -o $METABINDIR/metabat2 \
         -t "$SLURM_CPUS_PER_TASK" \
         -m 1500
if [ $? -eq 0 ]; then
            echo "metabat2 completed successfully for group: ${INPUT_GROUP}"
        else
            echo "metabat2 encountered an error for group: ${INPUT_GROUP}"
            exit 1  
        fi

##### run maxbin #####
module load uri/main
module load all/MaxBin/2.2.7-gompi-2021b
echo "activating maxbin2 env"
#run Maxbin2
run_MaxBin.pl -contig $CONTIGPATH/$CONTIGFILE \
            -reads $READSPATH/$F_READS \
            -reads2 $READSPATH/$R_READS \
            -thread "$SLURM_CPUS_PER_TASK" \
            -out $MAXBINDIR/maxbin2 
if [ $? -eq 0 ]; then
            echo "maxbin completed successfully for group: ${INPUT_GROUP}"
        else
            echo "maxbin encountered an error for group: ${INPUT_GROUP}"
            exit 1  
        fi

##### run CheckM on ALL #####
checkm lineage_wf -x fa -t "$SLURM_CPUS_PER_TASK" "$METABINDIR" "$METABINDIR/checkm-bins-stats"
checkm lineage_wf -x fasta -t "$SLURM_CPUS_PER_TASK" "$MAXBINDIR" "$MAXBINDIR/checkm-bins-stats"
checkm lineage_wf -x fa -t "$SLURM_CPUS_PER_TASK" "$CONCBINDIR" "$CONCBINDIR/checkm-bins-stats"

conda deactivate
module purge
echo "deactivating all env - script complete"

# JOB-ID: 
# bash script file name: brooke/seqs042024/binning.sh

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 20:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/slurm-%j.out  # %j = job ID  # %j = job ID

module load miniconda/22.11.1-1
conda activate binning

#set parameters for binning:
SPP="MCAV"
INPUT_GROUP = 
GROUP="healthy_2019_mcav"
OUTDIR=MetaBAT_"$GROUP"_bins2
DEPTHPATH=/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/MetaBAT_"$GROUP"_bins

CONTIGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}"

CONTIGFILE="${INPUT_GROUP}.contigs-fixed.fsa"

#run in dir: brooke/mcav/healthy_2019_mcav
mkdir ./binning/$OUTDIR

#create depth file for MetaBat2
#jgi_summarize_bam_contig_depths --outputDepth ./binning/$OUTDIR/MetaBAT_"$GROUP"_depth.txt $CONTIGPATH/index/*.bam

#MetaBat2 script with verbose output, minimum length (m)(has to be >=1500) and no min bin size 
metabat2 -i $CONTIGPATH/$CONTIGFILE -a $DEPTHPATH/MetaBAT_"$GROUP"_depth.txt \
-o ./binning/$OUTDIR/Metabat/ \
-m 1500 
-s 5000
#MetaBAT 2 (v2.12.1) using minContig 1500, minCV 1.0, minCVSum 1.0, 

# default parameters:
# -m [ --minContig ] arg (=2500), has to be at least 1500 
# maxP 95% Percentage of 'good' contigs considered for binning decided by connection among contigs
# minS 60 Minimum score of a edge for binning (should be between 1 and 99). The greater, the more specific.
# maxEdges 200 Maximum number of edges per node. The greater, the more sensitive.
#-x [ --minCV ] arg (=1)           Minimum mean coverage of a contig in each library for binning.
#  --minCVSum arg (=1)               Minimum total effective mean coverage of a contig (sum of depth over minCV) for binning.
# -s [ --minClsSize ] arg (=200000) Minimum size of a bin as the output.

#this runs CheckM immediately after and puts the results alongside your bins
checkm lineage_wf -x fa -t 3 ./binning/$OUTDIR/Metabat ./binning/$OUTDIR/Metabat/bins-stats

#bash script: metabat
#script location: brooke/mcav/healthy_2019_mcav/binning
#JOB ID: 19498217

 #### CONCOCT
https://concoct.readthedocs.io/en/latest/installation.html

This set of commands runs CONCOCT in its standard mode. It first creates a depth/coverage file for itself to use and then runs CONCOCT, with the standard settings. This means k-mer value is set to 4, minimum contig length is 1000, and CONCOCT runs on the exact amount of slots given to it by Hydra. 
 
CONCOCT creates a depth file out of the coverance created in the mapping step. It is key that this is all in the correct places before proceeding with binning. It creates a single file, which is then used for the complete binning process. Do keep in mind that binning might take awhile, so be prepared to let this run overnight.

In [ ]:
# Conda installation
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

conda create -n concoct_env python=3 concoct

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/slurm-%j-concoct.out  # %j = job ID  # %j = job ID

module load miniconda/22.11.1-1
conda activate concoct_env


#set parameters
SAMPLENAME="healthy_2019_mcav"
BINPATH=/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/Concoct_"$SAMPLENAME"_bins
CONTIGPATH=/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/mapping
CONTIGFILE="healthy_2019_mcav_filtered.contigs-fixed.fsa"
BAMPATH='/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/mapping/index'
TEMPDIR=concoct_"$SAMPLENAME"_temp

#mkdir $BINPATH
#creates the CONCOCT depth file
#this part cuts up the contigs into 10kb pieces for CONCOCT to use 
cut_up_fasta.py $CONTIGPATH/$CONTIGFILE -c 10000 -o 0 --merge_last -b $BINPATH/${SAMPLENAME}_contigs_cut.bed > $BINPATH/${SAMPLENAME}_contigs_cut.fa
#estimate contig coverage

concoct_coverage_table.py $BINPATH/${SAMPLENAME}_contigs_cut.bed $BAMPATH/*.bam > $BINPATH/coverage_table_${SAMPLENAME}.tsv || { echo 'Exit code 2: failed to create coverage file, exiting.' && exit; }

#CONCOCT script

#mkdir $BINPATH/$TEMPDIR

#run CONCOCT
concoct --composition_file $BINPATH/${SAMPLENAME}_contigs_cut.fa --coverage_file $BINPATH/coverage_table_${SAMPLENAME}.tsv -t 3 -b $BINPATH/$TEMPDIR || { echo 'Exit code 3: CONCOCT failed to run, exiting.' && exit; }
merge_cutup_clustering.py $BINPATH/$TEMPDIR/clustering_gt1000.csv > $BINPATH/$TEMPDIR/${SAMPLENAME}_clustering_merged.csv || { echo 'Exit code 4: failed to merge clusters, exiting.' && exit; }
extract_fasta_bins.py $CONTIGPATH/$CONTIGFILE $BINPATH/$TEMPDIR/${SAMPLENAME}_clustering_merged.csv --output_path $BINPATH || { echo 'Exit code 5: Bins were not extracted, exiting.' && exit; }

# Checkm is in binning env 
conda deactivate concoct_env
conda activate binning
#this runs CheckM immediately after and puts the results alongside your bins
checkm lineage_wf -x fa -t 3 $BINPATH  $BINPATH/CheckM
#can you add '--tab_table' to make output easier to read?

#bash script: concoct
#script location: mcav/healthy_2019_mcav/binning
#JOB ID: 21569803
#checkm job id: 21563395

### DAS_Tool 
- combines & refines bins into non-redundant set
https://github.com/cmks/DAS_Tool

In [33]:
import pandas as pd
import os

In [26]:
os.chdir('/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/Concoct_healthy_2019_mcav_bins/concoct_healthy_2019_mcav_temp')

In [40]:
# Convert contig-bin file from concoct to tab-delimited format
concoct_bins=pd.read_csv('healthy_2019_mcav_clustering_merged.csv',index_col=0)
concoct_bins
concoct_bins.to_csv("concoct_bins.tsv", sep='\t',header=False)

In [46]:
# Convert all contig-bin files from metabat to single tab-delimited file

directory = '/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/MetaBAT_healthy_2019_mcav_bins/Metabat'
output_file = os.path.join(directory, 'metabat_bins.tsv')
# Define a dictionary to store the mapping of contig IDs to bin IDs
contig_bin_mapping = {}

# Iterate through each file in the directory
for filename in os.listdir(directory):
    # Check if the file is a .fa file and matches the expected pattern
    if filename.endswith('.fa') and filename.startswith('.'):
        # Extract the bin ID from the filename
        bin_id = int(filename.split('.')[1])
        
        # Open the file and read the contig IDs
        with open(os.path.join(directory, filename), 'r') as file:
            for line in file:
                # Check if the line starts with '>'
                if line.startswith('>'):
                    # Extract the contig ID and add it to the dictionary with the corresponding bin ID
                    contig_id = line.strip().split()[0][1:]  # Remove '>' and any additional information
                    contig_bin_mapping[contig_id] = bin_id

# Write the contig IDs and corresponding bin IDs to a tab-delimited file
with open(output_file, 'w') as outfile:
    for contig_id, bin_id in contig_bin_mapping.items():
        outfile.write(f"{contig_id}\t{bin_id}\n")

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/slurm-%j-das_tool.out  # %j = job ID  # %j = job ID


module load miniconda/22.11.1-1
conda activate binning

# Set parameters
CONCOCTPATH='/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/Concoct_healthy_2019_mcav_bins/concoct_healthy_2019_mcav_temp/concoct_bins.tsv'  
METABATPATH='/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/binning/MetaBAT_healthy_2019_mcav_bins/Metabat/metabat_bins.tsv'
CONTIGPATH='/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/mcav/healthy_2019_mcav/mapping'
CONTIGFILE="healthy_2019_mcav_filtered.contigs-fixed.fsa"


DAS_Tool -i $CONCOCTPATH,$METABATPATH \
    -l concoct,metabat \
    -c $CONTIGPATH/$CONTIGFILE \
    -t 11 \
    --write_bin_evals \
    --write_bins \
    -o das_tool

# -i input list: tab seperated table of contigs-bins 
#--score_threshold default is 0.5

#bash script: das_tool
#script location: mcav/healthy_2019_mcav/binning
# JOBID: 21582072

### prep file to import as collection file to anvio 
- metabin produces fasta files containing contigs of each bin 
- collection artifact requires a txt file that contains list of contigs with their associated bins (2 columns) 

In [ ]:
#!/bin/bash
# A simple script to convert metabin results to anvio
FILES=$(find *.fa)
for f in $FILES; do
 NAME=$(basename $f .fasta)
 grep ">" $f | sed 's/>//' | sed -e "s/$/\t$NAME/" | sed 's/\./_/' >> metabins4anvio.txt
done